# Image Classification with MobileNetV2 Transfer Learning
## Apparel Images Dataset Classification

This notebook demonstrates a complete deep learning pipeline for classifying apparel images using MobileNetV2 with transfer learning. We'll:
- Load the Apparel Images Dataset from Kaggle
- Preprocess and augment the data
- Build a transfer learning model with MobileNetV2
- Train and evaluate the model
- Fine-tune specific layers for improved performance

**Author**: Deep Learning Pipeline Tutorial
**Goal**: Build a beginner-friendly, well-structured image classification pipeline

## Section 1: Import Required Libraries

Import all necessary libraries for building and training the deep learning model.

In [3]:
# Import required libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.pyplot import imshow
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

# PyTorch imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import StepLR
import torchvision
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from torchvision.models import MobileNet_V2_Weights

# Check device (GPU or CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch Version: {torch.__version__}")
print(f"Torchvision Version: {torchvision.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

ModuleNotFoundError: No module named 'pandas'

## Section 2: Load and Explore the Dataset

Load the Apparel Images Dataset from local folder structure. We'll analyze the dataset composition and display sample images.

In [ ]:
# Define the dataset path (customize this path to your local dataset location)
dataset_path = r'E:\daria\FS26\ANN\ANN_Project\project_ann\dataset'  # Change this to your dataset path

# Check if dataset exists
if os.path.exists(dataset_path):
    print(f"Dataset found at: {dataset_path}")
    print("\nDirectory structure:")
    for root, dirs, files in os.walk(dataset_path):
        level = root.replace(dataset_path, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        sub_indent = ' ' * 2 * (level + 1)
        for file in files[:3]:
            print(f'{sub_indent}{file}')
        if len(files) > 3:
            print(f'{sub_indent}... and {len(files) - 3} more files')
else:
    print(f"Dataset not found at {dataset_path}")
    print("Please download the dataset from: https://www.kaggle.com/datasets/trolukovich/apparel-images-dataset")
    print("And update the 'dataset_path' variable with the correct location.")

In [ ]:
# Count the number of images and classes in the dataset
def analyze_dataset(dataset_path):
    """
    Analyze the structure of the apparel dataset
    Returns: class_count dictionary with class names and number of images
    """
    class_count = {}
    
    for category in os.listdir(dataset_path):
        category_path = os.path.join(dataset_path, category)
        if os.path.isdir(category_path):
            image_files = [f for f in os.listdir(category_path) 
                          if f.lower().endswith(('.jpg', '.jpeg', '.png', '.gif'))]
            class_count[category] = len(image_files)
    
    return class_count

# Analyze dataset if it exists
if os.path.exists(dataset_path):
    class_counts = analyze_dataset(dataset_path)
    
    print("Dataset Analysis:")
    print(f"Total number of classes: {len(class_counts)}")
    print(f"\nClass distribution:")
    for class_name, count in sorted(class_counts.items()):
        print(f"  {class_name}: {count} images")
    
    total_images = sum(class_counts.values())
    print(f"\nTotal images: {total_images}")

In [ ]:
# Display sample images from each class
from PIL import Image

def display_sample_images(dataset_path, samples_per_class=2):
    """
    Display sample images from each clothing category
    """
    classes = os.listdir(dataset_path)
    classes = [c for c in classes if os.path.isdir(os.path.join(dataset_path, c))]
    
    num_classes = len(classes)
    fig, axes = plt.subplots(num_classes, samples_per_class, figsize=(12, 3 * num_classes))
    
    for idx, class_name in enumerate(sorted(classes)):
        class_path = os.path.join(dataset_path, class_name)
        images = [f for f in os.listdir(class_path) 
                 if f.lower().endswith(('.jpg', '.jpeg', '.png', '.gif'))]
        
        for sample_idx in range(min(samples_per_class, len(images))):
            ax = axes[idx, sample_idx] if num_classes > 1 else axes[sample_idx]
            img_path = os.path.join(class_path, images[sample_idx])
            
            try:
                img = Image.open(img_path)
                ax.imshow(img)
                ax.set_title(f'{class_name}')
                ax.axis('off')
            except Exception as e:
                print(f"Error loading {img_path}: {e}")
    
    plt.tight_layout()
    plt.show()

# Display samples if dataset exists
if os.path.exists(dataset_path):
    print("Displaying sample images from each clothing category:")
    display_sample_images(dataset_path, samples_per_class=2)

## Section 3: Data Preprocessing and Augmentation

Prepare the data by:
- Resizing images to 224x224 (MobileNetV2 standard input size)
- Normalizing pixel values
- Splitting into training (80%) and validation (20%) sets
- Applying data augmentation for better model robustness

In [ ]:
# Define image preprocessing parameters and transformations
IMG_HEIGHT = 224  # MobileNetV2 standard input height
IMG_WIDTH = 224   # MobileNetV2 standard input width
BATCH_SIZE = 32   # Number of images to process at once

# Training transformations: includes augmentation for better generalization
# Augmentation: rotation, horizontal flip, color jitter, etc.
train_transforms = transforms.Compose([
    transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),  # Resize to 224x224
    transforms.RandomRotation(20),                 # Random rotation up to 20 degrees
    transforms.RandomHorizontalFlip(0.5),          # 50% chance of horizontal flip
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),  # Color variations
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),  # Random translation
    transforms.ToTensor(),                         # Convert to tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet normalization
                        std=[0.229, 0.224, 0.225])
])

# Validation transformations: no augmentation, only resizing and normalization
val_transforms = transforms.Compose([
    transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

print("Data preprocessing configuration:")
print(f"  Image size: {IMG_HEIGHT} x {IMG_WIDTH}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Training transformations: Rotation, Flip, ColorJitter, Normalize")
print(f"  Validation transformations: Resize, Normalize")

In [ ]:
# Create a custom dataset class that splits data into train/validation
class ApparelDataset(Dataset):
    def __init__(self, dataset_path, transform=None, split='train', split_ratio=0.8):
        """
        Custom dataset for apparel images with train/validation split
        """
        self.dataset_path = dataset_path
        self.transform = transform
        self.split = split
        self.split_ratio = split_ratio
        self.images = []
        self.labels = []
        self.class_to_idx = {}
        
        # Load all images and create class indices
        class_names = sorted([d for d in os.listdir(dataset_path) 
                             if os.path.isdir(os.path.join(dataset_path, d))])
        
        for idx, class_name in enumerate(class_names):
            self.class_to_idx[class_name] = idx
            class_path = os.path.join(dataset_path, class_name)
            
            for img_file in os.listdir(class_path):
                if img_file.lower().endswith(('.jpg', '.jpeg', '.png', '.gif')):
                    self.images.append(os.path.join(class_path, img_file))
                    self.labels.append(idx)
        
        # Split into train/validation
        split_idx = int(len(self.images) * split_ratio)
        if split == 'train':
            self.images = self.images[:split_idx]
            self.labels = self.labels[:split_idx]
        else:  # validation
            self.images = self.images[split_idx:]
            self.labels = self.labels[split_idx:]
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            return torch.zeros(3, 224, 224), label

# Load datasets and create DataLoaders
if os.path.exists(dataset_path):
    train_dataset = ApparelDataset(dataset_path, transform=train_transforms, split='train')
    val_dataset = ApparelDataset(dataset_path, transform=val_transforms, split='val')
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    # Store class information
    class_to_idx = train_dataset.class_to_idx
    idx_to_class = {v: k for k, v in class_to_idx.items()}
    num_classes = len(class_to_idx)
    
    print(f"Successfully loaded data!")
    print(f"Number of classes: {num_classes}")
    print(f"Classes: {list(class_to_idx.keys())}")
    print(f"Training samples: {len(train_dataset)}")
    print(f"Validation samples: {len(val_dataset)}")
else:
    print("Dataset path not found. Please update dataset_path variable.")

## Section 4: Build the Transfer Learning Model

Transfer learning approach:
1. Load pretrained MobileNetV2 model (trained on ImageNet with 1.4M images)
2. Freeze base layers to retain learned features (weights won't be updated)
3. Add custom classification head for our apparel categories
4. Only train the new layers while using pre-learned features from the base

In [ ]:
# Load pretrained MobileNetV2 model
base_model = models.mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)

print(f"Base Model: MobileNetV2")
print(f"Model loaded with ImageNet pre-trained weights")
print(f"Model structure:")
print(base_model)

In [ ]:
# Freeze base model layers
# This prevents the pre-trained weights from being updated during training
for param in base_model.parameters():
    param.requires_grad = False

# Count trainable parameters
trainable_params = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in base_model.parameters())

print(f"\nBase model layer freeze status:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  All layers frozen: {trainable_params == 0}")

In [ ]:
# Build custom classification head
class TransferLearningModel(nn.Module):
    """
    Transfer Learning Model: MobileNetV2 base + custom classification head
    
    Architecture:
    - MobileNetV2 base (frozen): Feature extraction
    - Flatten layer: Convert spatial features to vector
    - Dense layer: Learn relationships between features
    - Dropout: Prevent overfitting
    - Output layer: Classification for num_classes
    """
    def __init__(self, base_model, num_classes):
        super(TransferLearningModel, self).__init__()
        
        # Use MobileNetV2 features without the final classification layer
        self.features = base_model.features
        
        # Custom classification head
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.flatten = nn.Flatten()
        
        # Dense layers for classification
        self.classifier = nn.Sequential(
            nn.Linear(1280, 256),        # 1280 is output of MobileNetV2 features
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            
            nn.Linear(128, num_classes)  # Output layer
        )
    
    def forward(self, x):
        # Feature extraction from base model
        x = self.features(x)
        
        # Global average pooling
        x = self.avgpool(x)
        
        # Flatten
        x = self.flatten(x)
        
        # Classification head
        x = self.classifier(x)
        
        return x

# Build the complete model
if os.path.exists(dataset_path):
    model = TransferLearningModel(base_model, num_classes)
    model = model.to(device)
    
    # Print model architecture
    print("Transfer Learning Model Architecture:")
    print(model)
    
    # Count trainable parameters in the model
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"\nModel Parameters:")
    print(f"  Total: {total_params:,}")
    print(f"  Trainable: {trainable_params:,}")

## Section 5: Compile and Train the Model

Configure the model for training and train it on our apparel dataset.

In [ ]:
# Compile model: define loss function and optimizer
if os.path.exists(dataset_path):
    # Loss function for multi-class classification
    criterion = nn.CrossEntropyLoss()
    
    # Optimizer: Adam with learning rate
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    print("Model compilation successful!")
    print("\nTraining configuration:")
    print(f"  Loss function: CrossEntropyLoss")
    print(f"  Optimizer: Adam (lr=0.001)")
    print(f"  Device: {device}")

In [ ]:
# Training function
def train_epoch(model, train_loader, criterion, optimizer, device):
    """
    Train for one epoch
    """
    model.train()
    running_loss = 0.0
    running_acc = 0.0
    
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Calculate accuracy
        _, predicted = torch.max(outputs.data, 1)
        running_loss += loss.item()
        running_acc += (predicted == labels).sum().item() / labels.size(0)
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = running_acc / len(train_loader)
    
    return epoch_loss, epoch_acc

# Validation function
def validate_epoch(model, val_loader, criterion, device):
    """
    Validate for one epoch
    """
    model.eval()
    running_loss = 0.0
    running_acc = 0.0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            _, predicted = torch.max(outputs.data, 1)
            running_loss += loss.item()
            running_acc += (predicted == labels).sum().item() / labels.size(0)
    
    epoch_loss = running_loss / len(val_loader)
    epoch_acc = running_acc / len(val_loader)
    
    return epoch_loss, epoch_acc

## Section 5: Train the Model

Configure the model for training and train it on our apparel dataset.

In [ ]:
# Train the model
if os.path.exists(dataset_path):
    num_epochs = 10
    history = {
        'train_loss': [],
        'val_loss': [],
        'train_acc': [],
        'val_acc': []
    }
    
    print("Starting training...\n")
    
    for epoch in range(num_epochs):
        # Train
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        
        # Validate
        val_loss, val_acc = validate_epoch(model, val_loader, criterion, device)
        
        # Store history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch [{epoch+1}/{num_epochs}]")
        print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    
    print("\nTraining complete!")
else:
    print("Dataset not found. Cannot train model.")
    history = None

In [ ]:
# Plot training history
if history:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    # Plot accuracy
    axes[0].plot(history['train_acc'], label='Training Accuracy', linewidth=2)
    axes[0].plot(history['val_acc'], label='Validation Accuracy', linewidth=2)
    axes[0].set_title('Model Accuracy over Epochs', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot loss
    axes[1].plot(history['train_loss'], label='Training Loss', linewidth=2)
    axes[1].plot(history['val_loss'], label='Validation Loss', linewidth=2)
    axes[1].set_title('Model Loss over Epochs', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("Training curves displayed successfully!")
else:
    print("No training history available.")

In [ ]:
# Generate predictions and confusion matrix
if os.path.exists(dataset_path):
    model.eval()
    
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    # Create confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    
    # Plot confusion matrix
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=list(idx_to_class.values()),
                yticklabels=list(idx_to_class.values()),
                cbar_kws={'label': 'Count'})
    plt.title('Confusion Matrix - Validation Set', fontsize=14, fontweight='bold')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
    
    # Print classification report
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds,
                               target_names=list(idx_to_class.values())))

## Section 7: Fine-tuning the Model (Optional)

Fine-tuning allows us to:
- Unfreeze some layers of the base model
- Train with a very low learning rate
- Allow the model to adapt to our specific apparel classification task
- This typically leads to better accuracy**Note:** This step requires more training time but can significantly improve performance.

In [ ]:
# Fine-tuning: Unfreeze some layers from the base model
if os.path.exists(dataset_path) and history:
    print("Starting fine-tuning process...\n")
    
    # Unfreeze the last N layers of the base model
    num_layers_to_unfreeze = 50
    
    for layer in base_model.features[-num_layers_to_unfreeze:]:
        for param in layer.parameters():
            param.requires_grad = True
    
    # Verify that some layers are now trainable
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Fine-tuning layer status:")
    print(f"  Total trainable parameters: {trainable_params:,}/{total_params:,}")
    
    # Recompile the optimizer with much lower learning rate
    optimizer = optim.Adam(model.parameters(), lr=0.0001)
    
    print(f"Model recompiled with lower learning rate (0.0001)")

In [ ]:
# Train the model with fine-tuning
if os.path.exists(dataset_path) and history:
    finetune_epochs = 5
    finetune_history = {
        'train_loss': [],
        'val_loss': [],
        'train_acc': [],
        'val_acc': []
    }
    
    print("Training with fine-tuning...\n")
    
    for epoch in range(finetune_epochs):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = validate_epoch(model, val_loader, criterion, device)
        
        finetune_history['train_loss'].append(train_loss)
        finetune_history['val_loss'].append(val_loss)
        finetune_history['train_acc'].append(train_acc)
        finetune_history['val_acc'].append(val_acc)
        
        print(f"Fine-tune Epoch [{epoch+1}/{finetune_epochs}]")
        print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    
    print("\nFine-tuning complete!")
else:
    print("Cannot perform fine-tuning. Dataset or initial training history not found.")
    finetune_history = None

In [ ]:
# Compare results before and after fine-tuning
if finetune_history and history:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    epochs_initial = len(history['train_acc'])
    epochs_finetune = len(finetune_history['train_acc'])
    total_epochs = epochs_initial + epochs_finetune
    
    # Accuracy plot
    axes[0].plot(range(epochs_initial), history['train_acc'], 
                label='Initial Training', linewidth=2)
    axes[0].plot(range(epochs_initial, total_epochs), finetune_history['train_acc'],
                label='Fine-tuning', linewidth=2)
    axes[0].axvline(x=epochs_initial, color='red', linestyle='--', alpha=0.5)
    axes[0].set_title('Accuracy: Transfer Learning vs Fine-tuning', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Loss plot
    axes[1].plot(range(epochs_initial), history['train_loss'],
                label='Initial Training', linewidth=2)
    axes[1].plot(range(epochs_initial, total_epochs), finetune_history['train_loss'],
                label='Fine-tuning', linewidth=2)
    axes[1].axvline(x=epochs_initial, color='red', linestyle='--', alpha=0.5)
    axes[1].set_title('Loss: Transfer Learning vs Fine-tuning', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\nComparison Summary:")
    print(f"Initial Training - Final Val Accuracy: {history['val_acc'][-1]:.4f}")
    print(f"Fine-tuning - Final Val Accuracy: {finetune_history['val_acc'][-1]:.4f}")
    improvement = (finetune_history['val_acc'][-1] - history['val_acc'][-1]) * 100
    print(f"Improvement: {improvement:+.2f}%")
else:
    print("Fine-tuning was not performed. Run fine-tuning cells to see comparison.")

## Summary and Next Steps

This notebook demonstrates a complete pipeline for apparel image classification using **PyTorch** and **MobileNetV2** transfer learning. Here's what we covered:

### What We Learned:
1. **Data Loading & Exploration** - Understanding the dataset structure using custom PyTorch Dataset class
2. **Preprocessing** - Resizing and normalizing images with torchvision transforms
3. **Transfer Learning** - Using pre-trained MobileNetV2 to leverage existing knowledge
4. **Model Training** - Training a custom classifier on top of frozen base layers with PyTorch
5. **Evaluation** - Assessing performance with accuracy, loss, and confusion matrix
6. **Fine-tuning** - Improving performance by training selected base layers

### Why PyTorch?
- **Flexible** - Easy to customize models and training loops
- **Dynamic computation graphs** - Debug code easily
- **Great ecosystem** - Excellent libraries for deep learning
- **Production-ready** - Easy to deploy with TorchScript or ONNX
- **Research-friendly** - Preferred by ML researchers

### Key Differences from TensorFlow:
- Custom Dataset classes for data loading (vs ImageDataGenerator)
- Manual training loops (vs fit())
- More flexibility and control over the training process

### Next Steps to Improve:
1. Increase training epochs (10+ for better convergence)
2. Implement learning rate scheduling (StepLR, CosineAnnealingLR)
3. Save best model checkpoint using torch.save()
4. Experiment with different optimizers (SGD, AdamW)
5. Try different base models (EfficientNet, ResNet50, ViT)
6. Implement mixed precision training for faster GPU training
7. Deploy model using TorchServe or ONNX

### Useful Resources:
- [PyTorch Documentation](https://pytorch.org/docs/stable/index.html)
- [PyTorch Transfer Learning Guide](https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)
- [Torchvision Models](https://pytorch.org/vision/stable/models.html)
- [Kaggle Apparel Dataset](https://www.kaggle.com/datasets/trolukovich/apparel-images-dataset)